# Fig S8 (v2) — AFR / EUR studies: identity LD, SER fallback and LD differences (data processing)

Rebuilds the identity-LD check (`FigS7.AFR_EUR_B_Identity_Check`, published as Figure S8) from the v2 AFR + EUR simulations (`33.Fig4C_4D_S9B_S9D.AFR_EUR_Meta_v2`), which use the unrelated 1000 Genomes AFR panel. The old check read the `28` B sweep, whose AFR panel held all 893 AFR including relatives; it is left as it was.

At AFR share 0% the study is a EUR study and at 100% an AFR study (n = 600 each); the mixtures (20%, 50%, 80%) are kept for the by-share summaries. Every simulation is fitted with the AFR panel LD, the EUR panel LD and the identity matrix, each with SuSiE-RSS and SuSiE-rRSS (`eb_mix`).

**Differences from the old check (read before comparing numbers):**
- `R_finite` is not shared across LD inputs: AFR LD uses the unrelated AFR panel size, EUR LD the 1,000-individual UK Biobank pool and identity 10,000 (`B_identity` in `33`). The old Figure S8 used 10,000 for both EUR LD and identity.
- The matched panel contains the study individuals (600 of the unrelated AFR, or 600 of the 1,000 EUR), so matched LD is close to in-sample LD.
- LD differences are against the in-sample LD of the sampled individuals (`_ld_diff`). The identity distance of the panels themselves (old Fig S8a) comes from the saved panel LD (`panel_LD/`).

**Inputs:** `AFR_EUR_meta_v2/summary/` (`[meta_v2_summary]`: the summary table and `_ld_diff`), `AFR_EUR_meta_v2/panel_LD/`

**Outputs (`figure_data/`):**
- `FigS8_v2.AFR_EUR_Identity_Check_Fallback` — SER fallback of each SuSiE-rRSS fit per AFR share × LD, pooled and per NCP, with the flag reason (Q_art > 0.1, D_CS,max ≥ 30) and median Q_art / D_CS,max
- `FigS8_v2.AFR_EUR_Identity_Check_Power_FDR` — power / FDR (bootstrap 95% CI) of SuSiE-RSS and the final SuSiE-rRSS (SER model when flagged) per AFR share × LD, pooled and per NCP
- `FigS8_v2.AFR_EUR_Identity_Check_LD_Difference` — per region and AFR share: MSE, max |ΔLD| and correlation of each LD input against the in-sample LD
- `FigS8_v2.AFR_EUR_Identity_Check_Panel_LD` — per region: MSE of the AFR and EUR panel LD against identity (mean off-diagonal r², all pairs, as old Fig S8a) and between the two panels

In [1]:
source("../../R/utils.R")
library(parallel)

data_name <- "FigS8_v2.AFR_EUR_Identity_Check"
job_dir   <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_meta_v2"
res_file  <- list.files(file.path(job_dir, "summary"), pattern = "_meta_v2_summary\\.rds$", full.names = TRUE)
stopifnot(length(res_file) == 1)
n_boot    <- 1000
q_art_cut <- 0.1      # reliability defaults: flagged when Q_art > 0.1 or D_CS,max >= 30
d_cs_cut  <- 30

## the three LD inputs, each with a SuSiE-RSS and a SuSiE-rRSS fit
ld_levels <- c("AFR LD", "EUR LD", "Identity")
ld_fits   <- tibble(LD   = ld_levels,
                    rss  = c("afr_refpanel", "eur_refpanel", "identity"),
                    rrss = c("afr_refpanel_rrss", "eur_refpanel_rrss", "identity_rrss"))
ld_of_method <- c(setNames(ld_fits$LD, ld_fits$rss), setNames(ld_fits$LD, ld_fits$rrss))

## AFR share 0% = EUR study, 100% = AFR study; for these two, the role of each LD relative to the study
study_label <- function(w) case_when(w == 0 ~ "EUR study", w == 1 ~ "AFR study", TRUE ~ paste0(round(100 * w), "% AFR"))
ld_role <- function(LD, w) case_when(LD == "Identity" ~ "Identity",
                                     (LD == "AFR LD" & w == 1) | (LD == "EUR LD" & w == 0) ~ "Matched",
                                     w %in% c(0, 1) ~ "Cross-population",
                                     TRUE ~ NA_character_)

## Simulations

In [2]:
res <- readRDS(res_file) %>%
  filter(str_remove(method, "_SER$") %in% c(ld_fits$rss, ld_fits$rrss)) %>%
  mutate(weight = as.numeric(prop))

## simulations per AFR share and NCP, and failed fits of each LD input
res %>% distinct(input_file, ncp, weight) %>% count(weight, ncp) %>%
  pivot_wider(names_from = ncp, values_from = n, names_prefix = "NCP ")
res %>% filter(!str_ends(method, "_SER"), !fit_ok) %>% count(method, weight, name = "n_failed_fits")

weight,NCP 20,NCP 40,NCP 80,NCP 120
<dbl>,<int>,<int>,<int>,<int>
0.0,1476,1467,1428,1376
0.2,1187,1198,1131,1096
0.5,1053,893,776,706
0.8,1344,1280,1132,992
1.0,1488,1484,1389,1311


method,weight,n_failed_fits
<chr>,<dbl>,<int>
afr_refpanel_rrss,1,1


## SER fallback

From the adjusted SuSiE-rRSS fit of each LD input. The reason is read from its `Q_art` and `D_CS_max` at the default cutoffs; the check below counts flags that neither value explains (expected 0).

In [3]:
base <- res %>% filter(!str_ends(method, "_SER"))
ser  <- res %>% filter(str_ends(method, "_SER")) %>% mutate(method = str_remove(method, "_SER$"))

fb_rows <- base %>%
  filter(method %in% ld_fits$rrss, fit_ok) %>%
  mutate(LD      = factor(ld_of_method[method], levels = ld_levels),
         flagged = R_reliability_flag %in% TRUE,
         by_Q    = !is.na(Q_art) & Q_art > q_art_cut,
         by_D    = !is.na(D_CS_max) & D_CS_max >= d_cs_cut,
         reason  = case_when(!flagged     ~ "Not flagged",
                             by_Q & by_D  ~ "Both",
                             by_Q         ~ "Q_art > 0.1",
                             by_D         ~ "D_CS,max >= 30",
                             TRUE         ~ "Other"))
## flags without a reason, and reasons without a flag (both expected 0)
fb_rows %>% summarise(flag_without_reason = sum(reason == "Other"), reason_without_flag = sum(!flagged & (by_Q | by_D)))

summarise_fb <- function(d) d %>%
  summarise(n_sim = n(), n_flagged = sum(flagged), fallback = n_flagged / n_sim,
            n_Q_only = sum(reason == "Q_art > 0.1"), n_D_only = sum(reason == "D_CS,max >= 30"), n_both = sum(reason == "Both"),
            Q_art_median = median(Q_art, na.rm = TRUE), D_CS_max_median = median(D_CS_max, na.rm = TRUE), .groups = "drop")
fallback <- bind_rows(
  fb_rows %>% group_by(weight, LD) %>% summarise_fb() %>% mutate(ncp = NA_real_),
  fb_rows %>% group_by(weight, LD, ncp) %>% summarise_fb()) %>%
  mutate(study = study_label(weight), role = ld_role(LD, weight), .after = weight)

## pure studies, pooled over NCP and causal counts
fallback %>% filter(is.na(ncp), weight %in% c(0, 1)) %>%
  transmute(study, LD, role, fallback = sprintf("%d/%d (%.1f%%)", n_flagged, n_sim, 100 * fallback),
            n_Q_only, n_D_only, n_both, Q_art_median, D_CS_max_median) %>% as.data.frame()
save_data(fallback, paste0(data_name, "_Fallback"))

flag_without_reason,reason_without_flag
<int>,<int>
0,0


study,LD,role,fallback,n_Q_only,n_D_only,n_both,Q_art_median,D_CS_max_median
<chr>,<fct>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>
EUR study,AFR LD,Cross-population,1328/5747 (23.1%),54,1227,47,0.051097166,9.1063913
EUR study,EUR LD,Matched,0/5747 (0.0%),0,0,0,0.007193052,0.5856906
EUR study,Identity,Identity,262/5747 (4.6%),0,262,0,0.000000000,1.8139269
AFR study,AFR LD,Matched,1/5671 (0.0%),0,1,0,0.003772153,0.9432449
AFR study,EUR LD,Cross-population,5367/5672 (94.6%),2042,834,2491,0.140359658,378.5701767
AFR study,Identity,Identity,293/5672 (5.2%),0,293,0,0.000000000,1.5508886


## Power and FDR

SuSiE-RSS and the final SuSiE-rRSS result (the SER model replaces a flagged fit, as Fig 4D v2). Bootstrap over simulations within each cell (`boot_cell_fast()` in `R/utils.R`).

In [4]:
use_ser <- ser %>%
  filter(method %in% ld_fits$rrss) %>%
  semi_join(filter(base, R_reliability_flag %in% TRUE), by = c("input_file", "method"))
final <- base %>%
  filter(method %in% c(ld_fits$rss, ld_fits$rrss)) %>%
  anti_join(select(use_ser, input_file, method), by = c("input_file", "method")) %>%
  bind_rows(use_ser) %>%
  filter(fit_ok) %>%
  mutate(model = factor(if_else(str_ends(method, "_rrss"), "SuSiE-rRSS", "SuSiE-RSS"), levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         LD    = factor(ld_of_method[method], levels = ld_levels))
stopifnot(!anyDuplicated(select(final, input_file, method)))

set.seed(1)
power_fdr <- bind_rows(
  final %>% group_by(model, LD, weight) %>% group_modify(~ boot_cell_fast(.x, n_boot)) %>% ungroup() %>% mutate(ncp = NA_real_),
  final %>% group_by(model, LD, weight, ncp) %>% group_modify(~ boot_cell_fast(.x, n_boot)) %>% ungroup()) %>%
  mutate(study = study_label(weight), role = ld_role(LD, weight), .after = weight) %>%
  pivot_longer(c(power, power_lo, power_hi, FDR, FDR_lo, FDR_hi), names_to = "key", values_to = "value") %>%
  mutate(metric = factor(if_else(str_starts(key, "power"), "Power", "FDR"), levels = c("Power", "FDR")),
         key    = case_when(str_ends(key, "_lo") ~ "lo", str_ends(key, "_hi") ~ "hi", TRUE ~ "est")) %>%
  pivot_wider(names_from = key, values_from = value)

## pure studies, pooled over NCP and causal counts
power_fdr %>% filter(is.na(ncp), weight %in% c(0, 1)) %>%
  transmute(study, LD, role, model, metric, value = sprintf("%.1f%% [%.1f, %.1f]", 100 * est, 100 * lo, 100 * hi)) %>%
  pivot_wider(names_from = metric, values_from = value) %>% arrange(study, model, LD) %>% as.data.frame()
save_data(power_fdr, paste0(data_name, "_Power_FDR"))

study,LD,role,model,Power,FDR
<chr>,<fct>,<chr>,<fct>,<chr>,<chr>
AFR study,AFR LD,Matched,SuSiE-RSS,"82.8% [81.8, 83.7]","0.5% [0.4, 0.7]"
AFR study,EUR LD,Cross-population,SuSiE-RSS,"61.7% [60.7, 62.7]","80.5% [80.1, 80.8]"
AFR study,Identity,Identity,SuSiE-RSS,"54.5% [53.5, 55.6]","72.7% [72.2, 73.2]"
AFR study,AFR LD,Matched,SuSiE-rRSS,"81.1% [80.1, 82.0]","0.5% [0.3, 0.6]"
AFR study,EUR LD,Cross-population,SuSiE-rRSS,"36.6% [35.8, 37.5]","1.6% [1.3, 2.1]"
AFR study,Identity,Identity,SuSiE-rRSS,"48.9% [47.9, 50.0]","68.2% [67.6, 68.8]"
EUR study,AFR LD,Cross-population,SuSiE-RSS,"46.7% [45.7, 47.7]","84.0% [83.6, 84.4]"
EUR study,EUR LD,Matched,SuSiE-RSS,"82.8% [81.9, 83.7]","2.3% [2.0, 2.6]"
EUR study,Identity,Identity,SuSiE-RSS,"38.0% [37.0, 39.1]","79.5% [79.0, 80.0]"


## LD difference from the in-sample LD

`pairs = "insample_r_gt_0.05"` uses pairs with |r| > 0.05 in the in-sample LD (as Fig 3B), `pairs = "all"` all upper-triangle pairs. The panel and identity references do not depend on z, so a region's simulations differ only in the sampled individuals; they are averaged to one value per region and AFR share. MSE against identity over all pairs is the mean off-diagonal r² of the in-sample LD; its correlation is undefined (`NA`).

In [5]:
ld_diff <- readRDS(sub("\\.rds$", "_ld_diff.rds", res_file)) %>%
  filter(reference %in% c("afr_refpanel", "eur_refpanel", "identity")) %>%
  mutate(weight = as.numeric(prop),
         LD     = factor(recode(reference, afr_refpanel = "AFR LD", eur_refpanel = "EUR LD", identity = "Identity"), levels = ld_levels))

ld_region <- ld_diff %>%
  group_by(region, weight, LD, pairs) %>%
  summarise(n_sim = n(), across(c(mse, max_abs_diff, concord, n_pairs), ~ mean(.x, na.rm = TRUE)), .groups = "drop") %>%
  mutate(across(c(mse, max_abs_diff, concord), ~ if_else(is.nan(.x), NA_real_, .x)),
         study = study_label(weight), role = ld_role(LD, weight), .after = weight)

## pure studies: median over regions
ld_region %>% filter(weight %in% c(0, 1)) %>%
  group_by(study, LD, role, pairs) %>%
  summarise(regions = n(), across(c(mse, max_abs_diff, concord), ~ signif(median(.x, na.rm = TRUE), 3)), .groups = "drop") %>%
  as.data.frame()

## identity-likeness of the in-sample LD (MSE vs identity, all pairs) in the AFR and EUR studies of the same region
ld_region %>% filter(LD == "Identity", pairs == "all", weight %in% c(0, 1)) %>%
  select(region, study, mse) %>% pivot_wider(names_from = study, values_from = mse) %>%
  filter(!is.na(`AFR study`), !is.na(`EUR study`)) %>%
  summarise(regions = n(), median_AFR = median(`AFR study`), median_EUR = median(`EUR study`),
            AFR_smaller = mean(`AFR study` < `EUR study`))
save_data(ld_region, paste0(data_name, "_LD_Difference"))

study,LD,role,pairs,regions,mse,max_abs_diff,concord
<chr>,<fct>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
AFR study,AFR LD,Matched,all,496,0.000124,0.0970,0.996
AFR study,AFR LD,Matched,insample_r_gt_0.05,496,0.000128,0.0877,0.998
AFR study,EUR LD,Cross-population,all,496,0.017800,1.2000,0.630
AFR study,EUR LD,Cross-population,insample_r_gt_0.05,496,0.031200,1.2000,0.685
AFR study,Identity,Identity,all,496,0.016200,1.0000,NA
AFR study,Identity,Identity,insample_r_gt_0.05,496,0.036200,1.0000,NA
EUR study,AFR LD,Cross-population,all,496,0.018300,1.2000,0.626
EUR study,AFR LD,Cross-population,insample_r_gt_0.05,496,0.034300,1.2000,0.670
EUR study,EUR LD,Matched,all,496,0.000634,0.1150,0.990


regions,median_AFR,median_EUR,AFR_smaller
<int>,<dbl>,<dbl>,<dbl>
496,0.01619779,0.03125048,1


## Identity distance of the panel LD (old Fig S8a)

Per region, from `panel_LD/EUR_AFR_<region>.panelLD.rds` (all variants polymorphic in both panels, written once per region by `33`): mean off-diagonal r² of the AFR and EUR panel LD (= MSE against identity over all pairs), and the MSE between the two panels.

In [6]:
panel_files <- list.files(file.path(job_dir, "panel_LD"), pattern = "\\.panelLD\\.rds$", full.names = TRUE)
read_panel <- function(f) {
  pl <- readRDS(f); ut <- upper.tri(pl$EUR_LD)
  a <- pl$AFR_LD[ut]; e <- pl$EUR_LD[ut]
  tibble(region = pl$region, p = length(pl$variant_id), B_AFR = pl$B_AFR, B_EUR = pl$B_EUR,
         AFR = mean(a^2, na.rm = TRUE), EUR = mean(e^2, na.rm = TRUE), cross_mse = mean((a - e)^2, na.rm = TRUE))
}
panel_ld <- bind_rows(mclapply(panel_files, function(f) tryCatch(read_panel(f), error = function(e) {
  message("FAILED: ", f, " : ", conditionMessage(e)); NULL }), mc.cores = 4))
cat(length(panel_files), "panel LD files,", nrow(panel_ld), "read; panel sizes: AFR",
    paste(unique(panel_ld$B_AFR), collapse = "/"), "and EUR", paste(unique(panel_ld$B_EUR), collapse = "/"), "\n")

panel_ld %>% summarise(regions = n(), median_AFR = median(AFR), median_EUR = median(EUR),
                       AFR_smaller = mean(AFR < EUR), median_cross_mse = median(cross_mse))
save_data(panel_ld, paste0(data_name, "_Panel_LD"))

496 panel LD files, 496 read; panel sizes: AFR 648 and EUR 1000 


regions,median_AFR,median_EUR,AFR_smaller,median_cross_mse
<int>,<dbl>,<dbl>,<dbl>,<dbl>
496,0.01606459,0.03062093,1,0.01765607
